### Tools

In [1]:
import os 
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [2]:
from langchain.chat_models import init_chat_model

model=init_chat_model(model="groq:qwen/qwen3-32b")
response=model.invoke("Why do parrot talk?")
response

AIMessage(content='<think>\nOkay, so why do parrots talk? I remember seeing parrots on TV or in videos, and they can mimic human speech. But why do they do that? Maybe it\'s just a habit they developed, or maybe there\'s a deeper reason. Let me think.\n\nFirst, I know that parrots are part of the Psittaciformes order. They have strong beaks and colorful feathers. They\'re also known for their intelligence. So maybe their ability to mimic speech is related to their intelligence. But other birds can mimic sounds too, like mynah birds or some songbirds. So maybe it\'s not just intelligence but something else.\n\nI think parrots are social animals. They live in groups, so maybe they talk to communicate with each other. But humans don\'t really know how they talk to each other. Wait, but when they mimic human speech, is it to communicate with humans? Like, maybe they\'re trying to fit in with their human companions by copying their sounds. That makes sense because animals often imitate the 

In [3]:
from  langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at the location"""
    return f"It's sunny in {location}"

model_with_tools=model.bind_tools([get_weather])

In [5]:
response=model_with_tools.invoke("Whats the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    print(f"Tool :{tool_call['name']}")
    print(f"Args :{tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking about the weather in Boston. I need to use the get_weather function. Let me check the function parameters. It requires a location, which in this case is Boston. I should structure the tool call with the location parameter set to "Boston". Make sure the JSON is correctly formatted. No other parameters are needed. Just call get_weather with location: Boston.\n', 'tool_calls': [{'id': 'fmg6h1s9y', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 102, 'prompt_tokens': 153, 'total_tokens': 255, 'completion_time': 0.168194926, 'completion_tokens_details': {'reasoning_tokens': 78}, 'prompt_time': 0.00605383, 'prompt_tokens_details': None, 'queue_time': 0.052701266, 'total_time': 0.174248756}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason': 'tool_c

In [9]:
from email import message
#Step 1:Model generates tool call
messages= [{"role":"user", "content":"What's the weather of Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

#Step 2 : Exicute tools and collect results
for tool_call in ai_msg.tool_calls :
    #Exicute the toolwith generated  argument
    tool_result=get_weather.invoke(tool_call)
    messages.append(tool_result)

#Step 3: Pass results back to model for final response
final_response= model_with_tools.invoke(messages)
print(final_response.text)


The weather in Boston is sunny right now. It's a great day to enjoy outdoor activities!
